# Feature Engineering

Notebook ini digunakan untuk menyiapkan model yang nantinya akan ditrain

# 1. Import Library

In [2]:
import pandas as pd
import numpy as np
import clickhouse_connect
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
from dotenv import load_dotenv
import os
import json
import warnings
warnings.filterwarnings('ignore')

load_dotenv('../.env')
print("✅ Library siap!")

✅ Library siap!


# 2. Koneksi & Ambil Data

In [5]:
client = clickhouse_connect.get_client(
    host='localhost',
    port=8123,
    username=os.getenv('CLICKHOUSE_USER', 'admin'),
    password=os.getenv('CLICKHOUSE_PASSWORD', 'admin123'),
    database='gold'
)

df = client.query_df("""
    SELECT 
        region,
        year,
        month,
        total_sales,
        total_profit,
        total_quantity,
        avg_discount,
        avg_sales_per_item
    FROM gold.forecast_training_monthly
    ORDER BY region, year, month
""")

print(f"✅ Data berhasil diambil: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head(10))

✅ Data berhasil diambil: 192 rows, 8 columns
    region  year  month  total_sales  total_profit  total_quantity  \
0  Central  2023      1    1539.9060      118.4902              62   
1  Central  2023      2    1233.1740      294.8067              69   
2  Central  2023      3    5827.6020     -274.0467             123   
3  Central  2023      4    3712.3400      229.3787              80   
4  Central  2023      5    4048.5060     -506.0171             121   
5  Central  2023      6    9646.2986     1246.1218             158   
6  Central  2023      7    6740.5740    -4055.3338             131   
7  Central  2023      8    3022.1830      628.4268              50   
8  Central  2023      9   34408.6898     1422.7972             264   
9  Central  2023     10    8965.7570     1341.2414             118   

   avg_discount  avg_sales_per_item  
0      0.157143               24.84  
1      0.284211               17.87  
2      0.206667               47.38  
3      0.147619               46

# 3. Eksplorasi Awal

In [6]:
print("=" * 50)
print("  INFO DATA")
print("=" * 50)
print(f"\nShape     : {df.shape}")
print(f"Regions   : {df['region'].unique()}")
print(f"Tahun     : {sorted(df['year'].unique())}")
print(f"\nMissing Values:\n{df.isnull().sum()}")
print(f"\nStatistik:\n{df[['total_sales','total_profit',
                            'total_quantity']].describe()}")

  INFO DATA

Shape     : (192, 8)
Regions   : <StringArray>
['Central', 'East', 'South', 'West']
Length: 4, dtype: string
Tahun     : [np.uint16(2023), np.uint16(2024), np.uint16(2025), np.uint16(2026)]

Missing Values:
region                0
year                  0
month                 0
total_sales           0
total_profit          0
total_quantity        0
avg_discount          0
avg_sales_per_item    0
dtype: int64

Statistik:
        total_sales  total_profit  total_quantity
count    192.000000    192.000000      192.000000
mean   12117.366429   1522.379243      201.322917
std     8749.770907   2090.411806      124.732467
min      199.776000  -4842.241900       15.000000
25%     5764.013375    360.696575      110.500000
50%     9730.607800   1204.503100      176.000000
75%    15703.335375   2347.744025      252.500000
max    45640.319000  10771.450700      643.000000


# 4. Encoding Region

In [7]:
le = LabelEncoder()
df['region_encoded'] = le.fit_transform(df['region'])

region_mapping = dict(zip(le.classes_, 
                          le.transform(le.classes_).tolist()))

print("✅ Region Encoding:")
for region, code in region_mapping.items():
    print(f"   {region} → {code}")

✅ Region Encoding:
   Central → 0
   East → 1
   South → 2
   West → 3


# 5. Lag Features

In [8]:
# Sort dulu sebelum buat lag — PENTING!
df = df.sort_values(['region', 'year', 'month']).reset_index(drop=True)

df['lag_1_sales'] = df.groupby('region')['total_sales'].shift(1)
df['lag_2_sales'] = df.groupby('region')['total_sales'].shift(2)
df['lag_3_sales'] = df.groupby('region')['total_sales'].shift(3)

print("✅ Lag features berhasil dibuat!")
print(df[['region','year','month','total_sales',
          'lag_1_sales','lag_2_sales','lag_3_sales']].head(10))

✅ Lag features berhasil dibuat!
    region  year  month  total_sales  lag_1_sales  lag_2_sales  lag_3_sales
0  Central  2023      1    1539.9060          NaN          NaN          NaN
1  Central  2023      2    1233.1740    1539.9060          NaN          NaN
2  Central  2023      3    5827.6020    1233.1740    1539.9060          NaN
3  Central  2023      4    3712.3400    5827.6020    1233.1740    1539.9060
4  Central  2023      5    4048.5060    3712.3400    5827.6020    1233.1740
5  Central  2023      6    9646.2986    4048.5060    3712.3400    5827.6020
6  Central  2023      7    6740.5740    9646.2986    4048.5060    3712.3400
7  Central  2023      8    3022.1830    6740.5740    9646.2986    4048.5060
8  Central  2023      9   34408.6898    3022.1830    6740.5740    9646.2986
9  Central  2023     10    8965.7570   34408.6898    3022.1830    6740.5740


# 6. Rolling Average Features

In [9]:
df['rolling_avg_3'] = df.groupby('region')['total_sales'] \
                        .transform(lambda x: x.shift(1).rolling(3).mean())

df['rolling_avg_6'] = df.groupby('region')['total_sales'] \
                        .transform(lambda x: x.shift(1).rolling(6).mean())

print("✅ Rolling average features berhasil dibuat!")
print(df[['region','year','month','total_sales',
          'rolling_avg_3','rolling_avg_6']].head(12))

✅ Rolling average features berhasil dibuat!
     region  year  month  total_sales  rolling_avg_3  rolling_avg_6
0   Central  2023      1    1539.9060            NaN            NaN
1   Central  2023      2    1233.1740            NaN            NaN
2   Central  2023      3    5827.6020            NaN            NaN
3   Central  2023      4    3712.3400    2866.894000            NaN
4   Central  2023      5    4048.5060    3591.038667            NaN
5   Central  2023      6    9646.2986    4529.482667            NaN
6   Central  2023      7    6740.5740    5802.381533    4334.637767
7   Central  2023      8    3022.1830    6811.792867    5201.415767
8   Central  2023      9   34408.6898    6469.685200    5499.583933
9   Central  2023     10    8965.7570   14723.815600   10263.098567
10  Central  2023     11   14057.5682   15465.543267   11138.668067
11  Central  2023     12   10635.5660   19144.005000   12806.845100


# 7. Hapus Baris NaN

In [10]:
df_before = len(df)
df = df.dropna().reset_index(drop=True)
df_after  = len(df)

print(f"✅ Rows sebelum drop NaN : {df_before}")
print(f"✅ Rows setelah drop NaN : {df_after}")
print(f"   Rows terhapus         : {df_before - df_after} (normal karena lag)")

✅ Rows sebelum drop NaN : 192
✅ Rows setelah drop NaN : 168
   Rows terhapus         : 24 (normal karena lag)


# 8. Susun Final Dataset

In [11]:
feature_cols = [
    'region',
    'region_encoded',
    'year',
    'month',
    'lag_1_sales',
    'lag_2_sales',
    'lag_3_sales',
    'rolling_avg_3',
    'rolling_avg_6',
    'avg_discount',
    'avg_sales_per_item',
    'total_quantity',
    'total_sales'       # ← TARGET (y)
]

df_final = df[feature_cols].copy()

print(f"✅ Final dataset shape: {df_final.shape}")
print(f"\nKolom: {list(df_final.columns)}")
print(f"\nSample data:")
print(df_final.head())

✅ Final dataset shape: (168, 13)

Kolom: ['region', 'region_encoded', 'year', 'month', 'lag_1_sales', 'lag_2_sales', 'lag_3_sales', 'rolling_avg_3', 'rolling_avg_6', 'avg_discount', 'avg_sales_per_item', 'total_quantity', 'total_sales']

Sample data:
    region  region_encoded  year  month  lag_1_sales  lag_2_sales  \
0  Central               0  2023      7    9646.2986    4048.5060   
1  Central               0  2023      8    6740.5740    9646.2986   
2  Central               0  2023      9    3022.1830    6740.5740   
3  Central               0  2023     10   34408.6898    3022.1830   
4  Central               0  2023     11    8965.7570   34408.6898   

   lag_3_sales  rolling_avg_3  rolling_avg_6  avg_discount  \
0    3712.3400    5802.381533    4334.637767      0.346875   
1    4048.5060    6811.792867    5201.415767      0.206250   
2    9646.2986    6469.685200    5499.583933      0.281176   
3    6740.5740   14723.815600   10263.098567      0.321875   
4    3022.1830   15465.5

# 9. Train/Test Split by Time

In [12]:
# 80% train, 20% test — berdasarkan WAKTU, bukan random!
cutoff_idx = int(len(df_final) * 0.8)
train_df   = df_final.iloc[:cutoff_idx]
test_df    = df_final.iloc[cutoff_idx:]

print(f"✅ Train set : {len(train_df)} rows")
print(f"   Periode   : "
      f"{train_df['year'].min()}-{int(train_df['month'].min()):02d} → "
      f"{train_df['year'].max()}-{int(train_df['month'].max()):02d}")

print(f"\n✅ Test set  : {len(test_df)} rows")
print(f"   Periode   : "
      f"{test_df['year'].min()}-{int(test_df['month'].min()):02d} → "
      f"{test_df['year'].max()}-{int(test_df['month'].max()):02d}")

✅ Train set : 134 rows
   Periode   : 2023-01 → 2026-12

✅ Test set  : 34 rows
   Periode   : 2024-01 → 2026-12


# 10. Define Evaluate Function

In [13]:
def evaluate_model(y_true, y_pred, model_name="Model"):
    """
    Fungsi evaluasi reusable untuk Person 3.
    
    Parameters:
        y_true     : nilai aktual (array/series)
        y_pred     : nilai prediksi (array/series)
        model_name : nama model untuk display
    
    Returns:
        dict berisi MAE, RMSE, MAPE
    """
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(
               np.abs((y_true - y_pred) / 
               np.where(y_true == 0, 1, y_true))
           ) * 100

    result = {
        'model' : model_name,
        'MAE'   : round(mae, 2),
        'RMSE'  : round(rmse, 2),
        'MAPE'  : round(mape, 2)
    }

    print(f"\n{'='*40}")
    print(f"  Evaluasi: {model_name}")
    print(f"{'='*40}")
    print(f"  MAE  : {mae:,.2f}")
    print(f"  RMSE : {rmse:,.2f}")
    print(f"  MAPE : {mape:.2f}%")
    print(f"{'='*40}")

    return result


def compare_models(results_list):
    """
    Bandingkan hasil evaluasi beberapa model sekaligus.
    
    Parameters:
        results_list : list of dict dari evaluate_model()
    
    Returns:
        DataFrame perbandingan model
    """
    df_compare = pd.DataFrame(results_list)
    df_compare = df_compare.sort_values('RMSE').reset_index(drop=True)

    print("\n📊 Model Comparison:")
    print(df_compare.to_string(index=False))
    print(f"\n🏆 Best Model : {df_compare.iloc[0]['model']}")
    print(f"   RMSE       : {df_compare.iloc[0]['RMSE']:,.2f}")
    print(f"   MAE        : {df_compare.iloc[0]['MAE']:,.2f}")
    print(f"   MAPE       : {df_compare.iloc[0]['MAPE']:.2f}%")

    return df_compare


print("✅ Fungsi evaluate_model() dan compare_models() siap!")
print("   → Kirim fungsi ini ke Person 3 untuk digunakan saat evaluasi model")

✅ Fungsi evaluate_model() dan compare_models() siap!
   → Kirim fungsi ini ke Person 3 untuk digunakan saat evaluasi model


# 11. Simpan Semua Output

In [ ]:
# Buat folder data kalau belum ada
os.makedirs('../data', exist_ok=True)

# Simpan CSV
train_df.to_csv('../data/forecast_train.csv', index=False)
test_df.to_csv('../data/forecast_test.csv',   index=False)
df_final.to_csv('../data/forecast_features.csv', index=False)

# Simpan region mapping
with open('../data/region_mapping.json', 'w') as f:
    json.dump(region_mapping, f, indent=2)

print("✅ Semua output berhasil disimpan!")
print("\n📄 File yang dihasilkan:")
print(f"   forecast_train.csv    → {len(train_df)} rows")
print(f"   forecast_test.csv     → {len(test_df)} rows")
print(f"   forecast_features.csv → {len(df_final)} rows")
print(f"   region_mapping.json   → {region_mapping}")

✅ Semua output berhasil disimpan!

📄 File yang dihasilkan:
   forecast_train.csv    → 134 rows
   forecast_test.csv     → 34 rows
   forecast_features.csv → 168 rows
   region_mapping.json   → {'Central': 0, 'East': 1, 'South': 2, 'West': 3}

📦 Siap dikirim ke Person 3!


# Notes
1. Pastikan sesudah pull lakukan dbt run pada terminal -> 'dbt run --select forecast_training_monthly'
2. Kemudian tambahkan library 'pip install scikit-learn python-dotenv clickhouse-connect'